# Recommendation System Analysis

This notebook provides an interactive environment to load the trained models, investigate the user embeddings, and manually review recommendations for given users.

In [ ]:
import os
import sys
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.join(os.getcwd(), '..'))
from src.config import MODELS_DIR
from src.hybrid_recommender import HybridRecommender

In [ ]:
# Load models
with open(os.path.join('..', MODELS_DIR, "preprocessor.pkl"), "rb") as f:
    preprocessor = pickle.load(f)
with open(os.path.join('..', MODELS_DIR, "cf_model.pkl"), "rb") as f:
    cf_model = pickle.load(f)
with open(os.path.join('..', MODELS_DIR, "cb_model.pkl"), "rb") as f:
    cb_model = pickle.load(f)
    
train_df = pd.read_pickle(os.path.join('..', MODELS_DIR, "train_df.pkl"))
test_df = pd.read_pickle(os.path.join('..', MODELS_DIR, "test_df.pkl"))
movies_df = pd.read_pickle(os.path.join('..', MODELS_DIR, "movies_df.pkl"))

user_counts = train_df['user_idx'].value_counts().to_dict()
hybrid_model = HybridRecommender(cf_model, cb_model, user_counts)

### View Blending Weights
Visualizing how the sigmoid function applies the Collaborative Filtering weight against interaction counts.

In [ ]:
history_lens = np.arange(0, 30)
weights = [hybrid_model._sigmoid_weight(n) for n in history_lens]

plt.figure(figsize=(10, 6))
plt.plot(history_lens, weights, marker='o', label='CF Weight')
plt.plot(history_lens, 1 - np.array(weights), marker='x', label='CB Weight')
plt.axvline(x=5, color='r', linestyle='--', label='Cold Start Threshold')
plt.title('Adaptive Blending Weights vs User Interaction Count')
plt.xlabel('Number of Interactions in Training')
plt.ylabel('Weight')
plt.legend()
plt.grid(True)
plt.show()